# Part 1: Exploring and Preparing the NYT Dataset

This notebook shows the main checks behind Part 1 of CS533 Homework 1. We explore the news articles, check the cleaning rules, inspect the time periods, and examine the official search collections and queries.

**P1:** 2005–2014. **P2:** 2015–2025 (2025 is incomplete).

The programs in `src/` created the actual training corpora and search files. **This notebook only reads data and displays checks. It does not write or replace any files.**

Run the cells from top to bottom. Loading and cleaning over one million articles can use substantial memory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path(r"C:\Projects\IR\CS533_HW1")
DATA_ROOT = REPO_ROOT.parent / "data"
ARTICLES_PATH = DATA_ROOT / "articles_mapped.h5"
OFFICIAL_DIR = DATA_ROOT / "nyt-tc-2026"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.monthly_counts import monthly_article_counts
from src.preprocessing import preprocess_text
from src.known_item_eligibility import check_known_item_eligibility

print("Article dataset found:", ARTICLES_PATH.is_file())
print("Official files found:", OFFICIAL_DIR.is_dir())

## 1. Explore the original dataset

First, we load the mapped article dataset once. We check its size, publication years, and categories to understand what we are working with **before cleaning**.

In [ ]:
df = pd.read_hdf(ARTICLES_PATH, key="data")

print(f"Original articles: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df[["web_url", "pub_date", "mapped_section"]].head(5))

# The year and period labels are used in later cells.
df["year"] = df["pub_date"].dt.year
df["period"] = df["year"].map(
    lambda y: "P1" if 2005 <= y <= 2014 else "P2" if 2015 <= y <= 2025 else "Outside"
)

In [ ]:
print("Articles by year:")
display(df["year"].value_counts().sort_index().rename("articles").to_frame())

print("Articles by category and period, before cleaning:")
display(pd.crosstab(df["mapped_section"], df["period"]))

### Check incomplete coverage

The dataset does not cover every month equally. We inspect 2009 and 2025, plus a sample of recent sports coverage, so we do not mistake gaps in the collection for genuine changes in news activity.

In [ ]:
print("Monthly article totals for 2009 and 2025:")
display(monthly_article_counts(df, years=[2009, 2025]))

sports_monthly = monthly_article_counts(
    df,
    years=[2023, 2024],
    categories=["Sports & Athletics"],
)
print("Sports coverage around late 2023:")
display(sports_monthly.loc["2023-07":"2024-06"])

## 2. Clean the articles

We apply the same three rules as the original Part 1 experiment, **in the same order**:

1. Remove text with fewer than 10 whitespace-separated tokens.
2. Keep the first article when identical text appears more than once.
3. Keep the first article when a URL appears more than once.

Each count is measured after the previous step, so the order matters. The original `df` remains unchanged; the result is stored in `clean_df`.

In [ ]:
short_mask = df["text"].fillna("").str.count(r"\S+") < 10
short_removed = int(short_mask.sum())
clean_df = df.loc[~short_mask].copy()

text_mask = clean_df["text"].duplicated(keep="first")
text_removed = int(text_mask.sum())
clean_df = clean_df.loc[~text_mask].copy()

url_mask = clean_df["web_url"].duplicated(keep="first")
url_removed = int(url_mask.sum())
clean_df = clean_df.loc[~url_mask].copy()

cleaning_summary = pd.DataFrame([
    {"Step": "Original", "Removed": 0, "Remaining": len(df)},
    {"Step": "Fewer than 10 tokens", "Removed": short_removed,
     "Remaining": len(df) - short_removed},
    {"Step": "Duplicate article text", "Removed": text_removed,
     "Remaining": len(df) - short_removed - text_removed},
    {"Step": "Duplicate URL", "Removed": url_removed, "Remaining": len(clean_df)},
])
display(cleaning_summary)
print(f"Final cleaned articles: {len(clean_df):,}")

In [ ]:
print("Cleaned article counts by period:")
display(clean_df["period"].value_counts().reindex(["P1", "P2"]).rename("articles").to_frame())

print("Cleaned articles by category and period:")
display(pd.crosstab(clean_df["mapped_section"], clean_df["period"]))

## 3. Preprocessing and the training corpora

Word2Vec and the later retrieval experiments need a consistent way to split text into words. We use the real `preprocess_text()` function from `src/preprocessing.py`, rather than recreating its regular expressions inside the notebook.

For example, the function lowercases words, normalizes dotted abbreviations, and keeps numbers. The production scripts used this preprocessing to create the P1 and P2 corpora.

In [ ]:
example = "The N.B.A. team's 2025 pre-season."
print("Original:", example)
print("Tokens:", preprocess_text(example))

print("\nFirst cleaned article, first 25 tokens:")
print(preprocess_text(clean_df["text"].iloc[0])[:25])

### Recorded corpus results

The corpus-writing and token-counting programs have **already been run**. Their verified totals are shown here so we do not regenerate large training files when rerunning this notebook.

| Period | Articles | Total tokens | Distinct tokens |
|---|---:|---:|---:|
| P1 (2005–2014) | 563,736 | 47,477,057 | 231,482 |
| P2 (2015–2025) | 406,984 | 30,206,984 | 181,582 |

The training corpora are stored outside Git in `data/corpora/`. The scripts `src/corpus_writer.py` and `src/token_counts.py` are responsible for producing and checking these numbers. This table is a record of the completed run, **not a new calculation**.

## 4. Inspect the official search collection

The official NYT-TC files provide the document identifiers, known-item queries, relevance judgments (qrels), and topic queries used in the retrieval part of the homework.

- A **document ID (`docno`)** identifies a searchable article.
- A **known-item query** is designed to find one specific article.
- A **qrel** records which document is relevant to a query.
- A **topic query** asks for articles about a broader subject, such as baseball.

In [ ]:
collection_p1 = pd.read_csv(OFFICIAL_DIR / "collection_P1.tsv", sep="\t")
collection_p2 = pd.read_csv(OFFICIAL_DIR / "collection_P2.tsv", sep="\t")
queries_p1 = pd.read_csv(OFFICIAL_DIR / "queries_ki_P1.tsv", sep="\t")
queries_p2 = pd.read_csv(OFFICIAL_DIR / "queries_ki_P2.tsv", sep="\t")
topics = pd.read_csv(OFFICIAL_DIR / "topics.tsv", sep="\t")

qrel_names = ["qid", "iteration", "docno", "relevance"]
qrels_p1 = pd.read_csv(OFFICIAL_DIR / "qrels_ki_P1.txt", sep=r"\s+", header=None, names=qrel_names)
qrels_p2 = pd.read_csv(OFFICIAL_DIR / "qrels_ki_P2.txt", sep=r"\s+", header=None, names=qrel_names)

summary = pd.DataFrame({
    "P1": [len(collection_p1), len(queries_p1), len(qrels_p1)],
    "P2": [len(collection_p2), len(queries_p2), len(qrels_p2)],
}, index=["Collection documents", "Known-item queries", "Known-item qrels"])
display(summary)
print("Number of topic queries:", len(topics))

In [ ]:
print("Sample P1 and P2 known-item queries:")
display(queries_p1[["qid", "query", "docno"]].head(3))
display(queries_p2[["qid", "query", "docno"]].head(3))

print("Sample topic queries:")
display(topics.head(5))

In [ ]:
print("Document counts per topic:")
topic_counts = pd.DataFrame({
    "P1": collection_p1["topic"].value_counts(),
    "P2": collection_p2["topic"].value_counts(),
}).fillna(0).astype(int)
display(topic_counts.sort_index())

print("Average words in a known-item query:")
print("P1:", round(queries_p1["query"].str.split().str.len().mean(), 2))
print("P2:", round(queries_p2["query"].str.split().str.len().mean(), 2))

## 5. Measure query-to-article word overlap

A known-item query should contain enough clues to find its target article. We measure **token overlap**: the fraction of query tokens that also appear in the matching cleaned article, after applying our preprocessing.

A higher percentage means that more query words appear in the article. This is useful for checking query quality, but it is **not** a retrieval accuracy score.

In [ ]:
def calculate_overlap(queries, collection, cleaned_articles):
    matched = (
        queries.merge(collection[["docno", "web_url"]], on="docno")
        .merge(cleaned_articles[["web_url", "text"]], on="web_url")
    )

    def overlap(row):
        query_tokens = preprocess_text(row["query"])
        if not query_tokens:
            return float("nan")
        article_tokens = set(preprocess_text(row["text"]))
        return sum(token in article_tokens for token in query_tokens) / len(query_tokens)

    matched["overlap"] = matched.apply(overlap, axis=1)
    return matched


matched_p1 = calculate_overlap(queries_p1, collection_p1, clean_df)
matched_p2 = calculate_overlap(queries_p2, collection_p2, clean_df)

print(f"P1: {len(matched_p1)} matches, average overlap {matched_p1['overlap'].mean():.2%}")
print(f"P2: {len(matched_p2)} matches, average overlap {matched_p2['overlap'].mean():.2%}")

## 6. Check eligibility for known-item query generation

Not every cleaned article can be used to construct a known-item query. The eligibility function applies the selection rules from our Part 1 implementation. We display the totals here; the actual query-generation and validation code remains in `src/` and `tests/`.

In [ ]:
eligibility = check_known_item_eligibility(clean_df)

eligible_count = int(eligibility["eligible"].sum())
ineligible_count = int((~eligibility["eligible"]).sum())
print(f"Eligible articles: {eligible_count:,}")
print(f"Ineligible articles: {ineligible_count:,}")

# No need to keep a second large article table in memory.
del eligibility

## 7. What Part 1 established

The original dataset contained **1,007,004 articles**. After removing short articles and duplicates, **970,720 remained**. The cleaned articles were divided into P1 and P2, then used to create the training corpora and retrieval collections.

The official material includes **1,000 known-item queries per period** and **32 topic queries**. Our earlier overlap analysis found averages of **86.51% for P1** and **83.50% for P2**.

These checks help us understand what data the Word2Vec experiments in Notebook 2 and the later retrieval systems are built on. Additional reconstruction checks and their saved results are documented in `report/part1_draft.md` and `data/part1_review/part1_results.json`.